# EUROPE SMALL CAP : pipeline des composites factoriels v2

Ce notebook teste deux configurations issues de l'analyse incrémentale SMALL CAP : `core_3` et `core_plus_satellites_5`. Chaque upgrade contient l'ancien facteur exact du screen et les variables sélectionnées dans l'analyse fournie. Toutes les composantes ont un poids relatif égal à 1,0 ; le poids normalisé est donc `1/(1+n)`, où `n` est le nombre de variables ajoutées.

La source fournie contient 2 119 tests sur la période totale uniquement. Le notebook revalide néanmoins les configurations sur chaque sous-période et sur le total. Il écrit uniquement sous `exports/` et ne contient aucune sortie d'exécution à sa création.


In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "func.py").exists():
    raise RuntimeError(
        "Le répertoire de travail Jupyter doit être la racine du projet."
    )

from func import (
    calculate_benchmark_performance,
    calculate_performance_ratios,
    combine_backtest_performances,
    export_backtest_results,
    load_backtest_data,
    plot_performance_comparison,
    test_composite_signals,
)
from factor_config import LOWER_IS_BETTER, signal_options

print("Les fonctions de recherche et de visualisation sont chargées.")


In [ ]:
MARKET = "EUROPE SMALL CAP"
BENCHMARK = "MSCI EUR SMALL"
START_DATE = "2007-12-01"
PERCENTILE = 0.13
N_JOBS = 1
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]
OUTPUT_NAME = "factor_family_pipeline_SMALL_v2"
EVIDENCE_REPORT = Path("exports") / "_agent_work" / "small_top12_audit.md"
SELECTION_SOURCE_NOTE = (
    "Analyse incrémentale SMALL CAP fournie par l'utilisateur : 2 119 tests "
    "sur la période totale, percentile 13 %, avec priorité au rendement actif, "
    "à l'IR, au worst CAGR, à la régularité relative et à la persistance."
)

BASELINE_COLUMNS = {
    "growth": "Growth Avg Percentile",
    "quality": "Quality Avg Percentile",
    "momentum": "Mom Avg Percentile",
    "value": "Value Avg Percentile",
    "dividend": "Dividend Avg Percentile",
}

CORE_SELECTIONS = {
    "dividend": [
        {"role": "core", "variable": "CFO Div Cov Ratio", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "total_performance", "evidence_note": "Meilleur compromis de la famille : active CAGR +0,327 point, IR +0,204, smoothness +4,21 et persistence +5,73 ; le risque absolu reste à contrôler."},
        {"role": "core", "variable": "DVD Yield FY1", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "total_performance", "evidence_note": "Améliore active CAGR et worst CAGR avec une smoothness relative positive ; le risque absolu n'est pas entièrement validé."},
        {"role": "core", "variable": "DVD Yield FY0", "dimension": "diff_6", "higher_is_better": True, "evidence_class": "total_performance", "evidence_note": "Variation lente du rendement, positive pour active CAGR, worst CAGR et persistence ; composante cyclique plutôt que court terme."},
    ],
    "growth": [
        {"role": "core", "variable": "5Y_Hist EPS TrendStab", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Active CAGR +4,37 points, IR +1,048 et worst CAGR +6,02 points, avec les deux contrôles de risque absolu favorables."},
        {"role": "core", "variable": "Gross Profit 5Y CAGR", "dimension": "rank_diff_3", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Très forte persistence (+22,37) et smoothness (+7,79), avec une amélioration de performance et de risque."},
        {"role": "core", "variable": "PCT Hist Sales", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Diversifie l'EPS par la dynamique historique des ventes : active CAGR +3,25 points, worst CAGR +5,27 points, risques favorables."},
    ],
    "momentum": [
        {"role": "core", "variable": "PCT ERR", "dimension": "level", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Meilleur signal de révision : active CAGR +1,98 point, IR +0,419, smoothness +3,22 et persistence +6,07, avec risque absolu favorable."},
        {"role": "core", "variable": "SP Price Target CIQ", "dimension": "pct_1", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Confirmation très récente de l'objectif de cours : active CAGR +1,65 point, IR +0,347 et smoothness +3,73, sans dégradation des risques testés."},
        {"role": "core", "variable": "EPS Med NTM -3M", "dimension": "pct_12", "higher_is_better": True, "evidence_class": "total_long_horizon", "evidence_note": "Horizon long qui complète les révisions courtes ; performance et régularité positives, mais le risque absolu doit être suivi."},
    ],
    "quality": [
        {"role": "core", "variable": "Oper Margin", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Plus forte contribution de rentabilité : active CAGR +5,94 points, IR +1,385 et worst CAGR +7,84 points, avec risques favorables."},
        {"role": "core", "variable": "Ebitda Margin", "dimension": "pct_3", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Améliore active CAGR de +4,53 points, IR de +1,072, smoothness de +8,39 et persistence de +17,26, avec risque absolu favorable."},
        {"role": "core", "variable": "Net Debt to Tot Equity", "dimension": "rank_diff_6", "higher_is_better": False, "evidence_class": "total_strict", "evidence_note": "Levier à horizon lent : active CAGR +2,90 points, worst CAGR +5,00 points, smoothness +11,39 et risques favorables."},
    ],
    "value": [
        {"role": "core", "variable": "Earns Yield FY0", "dimension": "level", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Gagnant net de la source : active CAGR +8,73 points, IR +1,613, worst CAGR +8,97 points, smoothness +24,68 et persistence +36,65."},
        {"role": "core", "variable": "PE LTM", "dimension": "rank_diff_3", "higher_is_better": False, "evidence_class": "total_strict", "evidence_note": "Amélioration du rang du multiple avec active CAGR +5,49 points, worst CAGR +8,30 points et persistence +26,93."},
        {"role": "core", "variable": "Price Cont Op Earning", "dimension": "rank_diff_3", "higher_is_better": False, "evidence_class": "total_strict", "evidence_note": "Deuxième confirmation de valorisation : active CAGR +5,35 points, worst CAGR +9,27 points et smoothness +12,33."},
    ],
}

SATELLITE_SELECTIONS = {
    "dividend": [
        {"role": "satellite_risk_watch", "variable": "CFO Div Cov Ratio", "dimension": "pct_1", "higher_is_better": True, "evidence_class": "total_smoothness", "evidence_note": "Variation courte de la couverture CFO : smoothness +3,87 et persistence +5,47 ; doublon économique assumé pour tester la complémentarité des horizons."},
        {"role": "satellite_risk_watch", "variable": "PCT DPS GR NTM", "dimension": "pct_3", "higher_is_better": True, "evidence_class": "total_risk_safe", "evidence_note": "Petite contribution mais les trois gains de performance sont positifs et les deux contrôles de risque absolu sont favorables."},
    ],
    "growth": [
        {"role": "satellite", "variable": "PCT Hist EPS", "dimension": "rank_diff_3", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Confirmation de la tendance EPS par le rang : persistence +22,12 et risques favorables."},
        {"role": "satellite", "variable": "Revenue 5Y CAGR", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Diversification par le chiffre d'affaires, avec smoothness +9,68 et risques favorables."},
    ],
    "momentum": [
        {"role": "satellite_risk_watch", "variable": "EPS Revision Ratio", "dimension": "level", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Niveau de révision complémentaire à PCT ERR : active CAGR +1,72 point, IR +0,352 et persistence +5,34, avec risques favorables."},
        {"role": "satellite_risk_watch", "variable": "Perf3M", "dimension": "pct_1", "higher_is_better": True, "evidence_class": "total_risk_safe", "evidence_note": "Petite composante tactique de prix ; gains positifs et risques absolus favorables, sans la faire dominer le momentum structurel."},
    ],
    "quality": [
        {"role": "satellite_risk_watch", "variable": "FCF Conversion", "dimension": "pct_6", "higher_is_better": True, "evidence_class": "total_smoothness", "evidence_note": "Contribution de rendement limitée mais smoothness +4,87, persistence +3,50 et risques absolus favorables ; test de diversification cash-flow."},
        {"role": "satellite_risk_watch", "variable": "Net Debt", "dimension": "diff_6", "higher_is_better": False, "evidence_class": "total_strict", "evidence_note": "Confirmation de désendettement : active CAGR +3,49 points, smoothness +10,71, persistence +17,06 et risques favorables."},
    ],
    "value": [
        {"role": "satellite", "variable": "Earns Yield FY1", "dimension": "level", "higher_is_better": True, "evidence_class": "total_strict", "evidence_note": "Forward earnings yield : active CAGR +6,77 points, IR +1,144, smoothness +15,25 et persistence +25,76."},
        {"role": "satellite", "variable": "EV To EBITDA LTM", "dimension": "pct_3", "higher_is_better": False, "evidence_class": "total_strict", "evidence_note": "Multiple EV/EBITDA en baisse favorable : active CAGR +5,13 points, worst CAGR +6,76 points et risques favorables."},
    ],
}

SELECTIONS_FULL = {
    family: list(CORE_SELECTIONS[family]) + list(SATELLITE_SELECTIONS[family])
    for family in CORE_SELECTIONS
}
SELECTION_VARIANTS = {
    "core_3": CORE_SELECTIONS,
    "core_plus_satellites_5": SELECTIONS_FULL,
}


## 1. Analyse de la source SMALL CAP et proposition de sélection

La source collée contient **2 119 lignes**, toutes sur `period_id=total` et `scope=total`, avec `PERCENTILE=0.13`. Les sélections ci-dessus sont donc des propositions de long horizon, pas encore une preuve de stabilité inter-régimes. Le notebook refait le test sur les sous-périodes afin de vérifier que le gain total ne vient pas d'une seule fenêtre.

La règle de sélection privilégie simultanément les deltas positifs de `active_cagr`, `top_information_ratio`, `top_worst_cagr`, `relative_smoothness_score` et `relative_persistence_score`. Les colonnes `incremental_risk_not_worse` et `incremental_absolute_risk_not_worse` sont contrôlées séparément. Un delta de `0.01` d'active CAGR représente environ **1 point de pourcentage annualisé**, tandis qu'un delta de `+4` de smoothness représente quatre points du score relatif.

| Famille | Lecture de la source | Proposition principale |
|---|---|---|
| Dividend | Les meilleurs rendements sont positifs, mais les candidats à plus forte contribution ne passent pas tous le risque absolu. | Couverture CFO et rendement du dividende en core ; croissance du DPS et horizon court en satellites de surveillance. |
| Growth | `5Y_Hist EPS TrendStab`, `Gross Profit 5Y CAGR` et `PCT Hist Sales` améliorent à la fois performance et risques testés. | Mélange EPS, gross profit et ventes, puis confirmations EPS/revenue. |
| Momentum | `PCT ERR`, `EPS Revision Ratio` et `SP Price Target CIQ` dominent les signaux de révision et de cible. | Révisions earnings en core, cible de cours et prix court en satellites. |
| Quality | Rentabilité opérationnelle et désendettement sont les contributions les plus nettes ; `FCF Conversion` est surtout une diversification de régularité. | Marges et levier en core ; cash-flow et dette en satellites. |
| Value | `Earns Yield FY0` est le signal le plus fort de tout le fichier ; les multiples et le forward yield confirment. | Earnings yield spot, PE et prix opérationnel en core ; forward yield et EV/EBITDA en satellites. |

`core_3` ajoute trois variables au facteur historique ; `core_plus_satellites_5` en ajoute cinq. L'ancien facteur exact du screen est inclus comme une composante supplémentaire de poids 1,0 : les poids normalisés sont donc 25 % ou 16,67 %. Les satellites ne sont pas une garantie de performance : ils servent à tester la diversification, la smoothness relative et le risque absolu.

Les seuls facteurs historiques autorisés sont `Growth Avg Percentile`, `Mom Avg Percentile`, `Value Avg Percentile`, `Quality Avg Percentile` et `Dividend Avg Percentile`. Aucune colonne de secours n'est utilisée.


In [ ]:
def validate_selection_directions(selections):
    """Vérifie la direction économique de chaque variable brute."""
    errors = []
    for family, specs in selections.items():
        for spec in specs:
            expected_higher = spec["variable"] not in LOWER_IS_BETTER
            if bool(spec["higher_is_better"]) != expected_higher:
                expected_label = "higher" if expected_higher else "lower"
                errors.append(
                    f"{family}: {spec['variable']} doit être {expected_label}-is-better"
                )
    if errors:
        raise ValueError(
            "Directions incompatibles avec factor_config.LOWER_IS_BETTER : "
            + "; ".join(errors)
        )


for variant_name, selections in SELECTION_VARIANTS.items():
    validate_selection_directions(selections)
    for family, specs in selections.items():
        if len(specs) not in (3, 5):
            raise ValueError(
                f"{variant_name}/{family} doit contenir 3 ou 5 variables candidates."
            )


def make_baseline_config(family, weight=1.0):
    variable = BASELINE_COLUMNS[family]
    return {variable: signal_options(level=float(weight), higher_is_better=True)}


def make_family_config(family, specs):
    """Construit un upgrade avec l'ancien facteur et les variables ajoutées."""
    config = make_baseline_config(family, weight=1.0)
    for spec in specs:
        variable = spec["variable"]
        if variable not in config:
            config[variable] = signal_options(
                higher_is_better=bool(spec["higher_is_better"])
            )
        config[variable][f"weight_{spec['dimension']}"] = 1.0
    return config


SELECTION_ROWS = []
for variant_name, selections in SELECTION_VARIANTS.items():
    for family, specs in selections.items():
        equal_weight = 1.0 / (1.0 + len(specs))
        for index, spec in enumerate(specs, start=1):
            SELECTION_ROWS.append(
                {
                    "market": MARKET,
                    "variant": variant_name,
                    "family": family,
                    "component_index": index,
                    "role": spec["role"],
                    "variable": spec["variable"],
                    "dimension": spec["dimension"],
                    "higher_is_better": spec["higher_is_better"],
                    "evidence_class": spec.get("evidence_class", spec["role"]),
                    "evidence_note": spec.get("evidence_note", "Validation conjointe requise après les sous-périodes."),
                    "source_report": str(EVIDENCE_REPORT),
                    "selection_source_note": SELECTION_SOURCE_NOTE,
                    "raw_component_weight": 1.0,
                    "equal_component_weight_with_old_factor": equal_weight,
                    "baseline_column": None,
                }
            )
SELECTION_MANIFEST = pd.DataFrame(SELECTION_ROWS)
display(SELECTION_MANIFEST)


In [ ]:

DATA_DIR = REPO_ROOT / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregateCIQ.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"
EXPORT_ROOT = REPO_ROOT / "exports"
EXPORT_DIR = EXPORT_ROOT / OUTPUT_NAME
LIST_NOIRE_PATH = None

try:
    import pyarrow.parquet as pq
    available_columns = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
except Exception as error:
    raise RuntimeError(
        "Échec de lecture du schéma parquet du screen ; vérifiez que pyarrow est disponible."
    ) from error

missing_baselines = [
    column for column in BASELINE_COLUMNS.values()
    if column not in available_columns
]
if missing_baselines:
    raise KeyError(
        "Les anciens facteurs obligatoires sont absents du screen : "
        f"{missing_baselines}"
    )

SELECTION_MANIFEST["baseline_column"] = SELECTION_MANIFEST["family"].map(BASELINE_COLUMNS)
SELECTION_MANIFEST["baseline_included_in_composite"] = True

SELECTED_RAW_VARIABLES = sorted(
    {
        spec["variable"]
        for selections in SELECTION_VARIANTS.values()
        for specs in selections.values()
        for spec in specs
    }
)
LOAD_VARIABLES = list(
    dict.fromkeys(SELECTED_RAW_VARIABLES + list(BASELINE_COLUMNS.values()))
)

screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=LOAD_VARIABLES,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

missing = [column for column in LOAD_VARIABLES if column not in screen.columns]
if missing:
    raise KeyError(f"Variables absentes après chargement : {missing}")
if f"Weight in {BENCHMARK}" not in screen.columns:
    raise KeyError(f"La colonne Weight in {BENCHMARK} est absente du screen")

BENCH_PERF = calculate_benchmark_performance(
    screen=screen,
    returns=returns,
    bench=BENCHMARK,
    start_date=START_DATE,
)

MONTHLY_BASE_CACHE = {}
RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": BENCH_PERF,
    "percentile": PERCENTILE,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": MONTHLY_BASE_CACHE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": False,
    "build_figure": False,
}

print(f"screen={screen.shape}, returns={returns.shape}")
print(f"benchmark={BENCHMARK}; baseline columns={BASELINE_COLUMNS}")
print("Chaque upgrade comprend l'ancien facteur et des composantes de poids relatif égal.")


## 2. Construire et backtester les nouveaux composites familiaux et les facteurs existants du screen

In [ ]:
COMPOSITE_CONFIGS = {}
for variant_name, selections in SELECTION_VARIANTS.items():
    for family, specs in selections.items():
        COMPOSITE_CONFIGS[f"{variant_name}__{family}"] = make_family_config(
            family, specs
        )
for family in CORE_SELECTIONS:
    COMPOSITE_CONFIGS[f"screen_baseline_{family}"] = make_baseline_config(family)

composite_batch = test_composite_signals(
    screen=screen,
    returns=returns,
    composite_configs=COMPOSITE_CONFIGS,
    list_noire_path=LIST_NOIRE_PATH,
    score_prefix="Score_FamilyPipelineSmallV2",
    **RUN_OPTIONS,
)
screen = composite_batch["screen"]
print(
    "Le backtest des deux upgrades et des facteurs historiques du screen est terminé."
)


## 3. Contrôle des deux upgrades

Pour chaque famille, `core_3` contient l'ancien facteur et trois ajouts ; `core_plus_satellites_5` contient l'ancien facteur et cinq ajouts. Les figures et les exports affichent séparément l'ancien facteur (`Ancien facteur`) et le benchmark afin de distinguer l'amélioration de l'upgrade de la simple performance absolue.

In [ ]:

all_results = {
    "composite_comparison": composite_batch,
}
print("Les backtests des composites familiaux et des facteurs existants du screen sont terminés.")


## 4. Export normalisé

L'export conserve les métriques officielles, les courbes de performance, la comparaison de chaque upgrade avec le facteur historique et le manifest détaillé des dimensions testées. Lire d'abord les deltas de CAGR actif, d'IR et de worst CAGR ; contrôler ensuite le drawdown actif, le tracking error et les périodes économiques.

In [ ]:

exported = export_backtest_results(
    results=all_results,
    output_dir=EXPORT_ROOT,
    export_name=OUTPUT_NAME,
    export_html=False,
    export_png=False,
    export_holdings=False,
)
EXPORT_DIR = Path(exported["export_dir"])

metrics = pd.read_csv(EXPORT_DIR / "backtest_metrics.csv")
with (EXPORT_DIR / "backtest_registry.json").open("r", encoding="utf-8") as handle:
    registry = json.load(handle)
path_by_name = {
    entry.get("metadata", {}).get("test_name"): entry.get("test_path")
    for entry in registry
    if entry.get("metadata", {}).get("test_name") and entry.get("test_path")
}

METRIC_COLUMNS = [
    "active_cagr",
    "top_worst_cagr",
    "top_information_ratio",
    "robust_score",
    "active_max_drawdown",
    "tracking_error_annualized",
    "min_rolling_3y_cagr",
    "top_bench_ratio",
    "top_worst_ratio",
    "top_annualized_return",
    "bench_annualized_return",
    "observation_count",
    "years",
]
COMPARABILITY_COLUMNS = [
    "robust_score_comparable"
] if "robust_score_comparable" in metrics.columns else []


def _path_for(test_name):
    if test_name not in path_by_name:
        raise KeyError(f"Le test_name={test_name} est absent du registry exporté")
    return path_by_name[test_name]


def _metric_slice(test_path):
    return metrics.loc[metrics["test_path"].eq(test_path)].copy()


family_comparison_parts = []
family_names = list(CORE_SELECTIONS)
for variant_name, selections in SELECTION_VARIANTS.items():
    for family in family_names:
        new_rows = _metric_slice(_path_for(f"{variant_name}__{family}"))
        base_rows = _metric_slice(_path_for(f"screen_baseline_{family}"))
        left_columns = ["period_id", "scope", "period_label", *METRIC_COLUMNS, *COMPARABILITY_COLUMNS]
        right_columns = ["period_id", "scope", *METRIC_COLUMNS, *COMPARABILITY_COLUMNS]
        left = new_rows[left_columns].rename(
            columns={column: f"{column}_new" for column in [*METRIC_COLUMNS, *COMPARABILITY_COLUMNS]}
        )
        right = base_rows[right_columns].rename(
            columns={column: f"{column}_screen" for column in [*METRIC_COLUMNS, *COMPARABILITY_COLUMNS]}
        )
        joined = left.merge(right, on=["period_id", "scope"], how="outer")
        joined.insert(0, "family", family)
        joined.insert(0, "variant", variant_name)
        for column in METRIC_COLUMNS:
            joined[f"delta_{column}"] = (
                joined[f"{column}_new"] - joined[f"{column}_screen"]
            )
        joined["new_perf_gate"] = (
            joined["active_cagr_new"].gt(0)
            & joined["top_worst_cagr_new"].gt(0)
            & joined["top_information_ratio_new"].gt(0)
        )
        joined["screen_perf_gate"] = (
            joined["active_cagr_screen"].gt(0)
            & joined["top_worst_cagr_screen"].gt(0)
            & joined["top_information_ratio_screen"].gt(0)
        )
        joined["performance_improved"] = (
            joined["delta_active_cagr"].gt(0)
            & joined["delta_top_worst_cagr"].gt(0)
            & joined["delta_top_information_ratio"].gt(0)
        )
        joined["risk_not_worse"] = (
            joined["delta_active_max_drawdown"].le(0)
            & joined["delta_tracking_error_annualized"].le(0)
        )
        if COMPARABILITY_COLUMNS:
            comparable_period = (
                joined["scope"].eq("total")
                | (
                    joined["robust_score_comparable_new"].astype(str).str.lower().isin(["true", "1", "yes"])
                    & joined["robust_score_comparable_screen"].astype(str).str.lower().isin(["true", "1", "yes"])
                )
            )
        else:
            comparable_period = joined["scope"].eq("total")
        joined["strict_comparable_improvement"] = (
            comparable_period
            & joined["performance_improved"]
            & joined["risk_not_worse"]
            & joined["robust_score_new"].gt(joined["robust_score_screen"])
        )
        family_comparison_parts.append(joined)

family_comparison = pd.concat(family_comparison_parts, ignore_index=True)
family_comparison.to_csv(
    EXPORT_DIR / "family_composite_vs_screen.csv", index=False
)
family_comparison.loc[family_comparison["period_id"].eq("total")].to_csv(
    EXPORT_DIR / "family_composite_vs_screen_total.csv", index=False
)

performance_selections = {}
first_baseline_path = None
for variant_name in SELECTION_VARIANTS:
    for family in family_names:
        new_path = _path_for(f"{variant_name}__{family}")
        performance_selections[f"{variant_name}__{family}"] = (new_path, "Top")
for family in family_names:
    base_path = _path_for(f"screen_baseline_{family}")
    performance_selections[f"screen__{family}"] = (base_path, "Top")
    first_baseline_path = first_baseline_path or base_path
performance_selections["Benchmark"] = (first_baseline_path, "Bench")

top_curves = combine_backtest_performances(
    export_dir=EXPORT_DIR,
    selections=performance_selections,
)
top_curves.to_csv(EXPORT_DIR / "performance_top_curves.csv", index=True)
top_ratios = calculate_performance_ratios(top_curves, benchmark_column="Benchmark")
top_ratios.to_csv(EXPORT_DIR / "performance_ratios.csv", index=True)

selection_figure_paths = {}
selection_figure_data = {}
for family in family_names:
    family_selections = {
        variant_name: (f"{variant_name}__{family}", "Top")
        for variant_name in SELECTION_VARIANTS
    }
    family_selections["Ancien facteur"] = (f"screen_baseline_{family}", "Top")
    family_selections["Benchmark"] = (f"screen_baseline_{family}", "Bench")
    family_performance = combine_backtest_performances(
        export_dir=EXPORT_DIR,
        selections=family_selections,
    )
    family_ratios = calculate_performance_ratios(
        family_performance,
        benchmark_column="Benchmark",
    )
    family_performance.to_csv(
        EXPORT_DIR / f"performance_{family}_selections.csv",
        index=True,
    )
    family_ratios.to_csv(
        EXPORT_DIR / f"performance_{family}_selection_ratios.csv",
        index=True,
    )
    figure_path = EXPORT_DIR / "figures" / f"selection_comparison_{family}.html"
    family_figure = plot_performance_comparison(
        performance=family_performance,
        ratios=family_ratios,
        benchmark_column="Benchmark",
        title=f"EUROPE SMALL CAP | {family} | deux configurations",
        save_path=figure_path,
        show_plot=False,
        rebase=True,
        period_breakpoints=PERIOD_BREAKPOINTS,
        show_worst_performance=False,
    )
    selection_figure_paths[family] = str(figure_path)
    selection_figure_data[family] = {"performance": family_performance, "ratios": family_ratios}
    display(family_figure)

run_manifest = {
    "market": MARKET,
    "benchmark": BENCHMARK,
    "start_date": START_DATE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "percentile": PERCENTILE,
    "rebalancing_frequency": 1,
    "fill_method": "copy",
    "baseline_columns": BASELINE_COLUMNS,
    "selection_variants": list(SELECTION_VARIANTS),
    "selection_figure_paths": selection_figure_paths,
    "selection_manifest": SELECTION_ROWS,
    "source_evidence_report": str(EVIDENCE_REPORT),
    "selection_source_note": SELECTION_SOURCE_NOTE,
    "baseline_included_in_composites": True,
    "old_factor_displayed_in_family_figures": True,
    "equal_weight_rule": "1/(1+n_additions)",
    "gate": {
        "performance": "active_cagr > 0 and top_worst_cagr > 0 and top_information_ratio > 0",
        "strict_comparable": "performance gate and robust_score > 0",
        "short_period_note": "robust_score is diagnostic when robust_score_comparable is false",
    },
    "outputs": [
        "backtest_metrics.csv",
        "backtest_registry.json",
        "family_composite_vs_screen.csv",
        "family_composite_vs_screen_total.csv",
        "performance_top_curves.csv",
        "performance_ratios.csv",
        "performance_<family>_selections.csv",
        "performance_<family>_selection_ratios.csv",
        "figures/selection_comparison_<family>.html",
        "selection_manifest.csv",
        "run_manifest.json",
    ],
}
SELECTION_MANIFEST.to_csv(EXPORT_DIR / "selection_manifest.csv", index=False)
with (EXPORT_DIR / "run_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)

print(f"Répertoire des résultats : {EXPORT_DIR}")
display(
    family_comparison.loc[
        family_comparison["period_id"].eq("total"),
        [
            "variant",
            "family",
            "active_cagr_new",
            "active_cagr_screen",
            "delta_active_cagr",
            "top_information_ratio_new",
            "top_information_ratio_screen",
            "delta_top_information_ratio",
            "robust_score_new",
            "robust_score_screen",
            "strict_comparable_improvement",
        ],
    ].sort_values("delta_active_cagr", ascending=False)
)


## 5. Ordre de lecture après exécution

1. Consulter `selection_manifest.csv` pour vérifier chaque variable et sa dimension exacte (`level`, `diff_N`, `pct_N`, `rank_diff_N`), ainsi que le poids normalisé avec l'ancien facteur.
2. Lire `family_composite_vs_screen_total.csv` et comparer `core_3` et `core_plus_satellites_5` à l'ancien facteur du screen.
3. Lire ensuite les lignes par période de `family_composite_vs_screen.csv` ; une amélioration totale ne suffit pas si elle vient d'une seule période.
4. Utiliser `performance_<family>_selection_ratios.csv` et les figures : chaque figure contient maintenant l'ancien facteur, les deux upgrades et le benchmark.

Les résultats de cette version sont une validation conjointe des sélections ; ils ne remplacent pas l'analyse incrémentale variable par variable. Les satellites Momentum et Quality doivent faire l'objet d'un contrôle spécifique de risque absolu.